In [1]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from scipy.stats import binomtest

In [2]:
def c2st(P, Q, test_size=0.5, seed=42):
    # Combine both samples
    X = np.concatenate([P, Q]).reshape(-1, 1)

    # Label P as 0 and Q as 1
    y = np.concatenate([
        np.zeros(len(P), dtype=int),
        np.ones(len(Q), dtype=int)
    ])

    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=seed,
        stratify=y
    )

    # Train binary classifier
    classifier = LogisticRegression()
    classifier.fit(X_train, y_train)

    # Predict on unseen test data
    predictions = classifier.predict(X_test)

    # Calculate accuracy
    correct = np.sum(predictions == y_test)
    accuracy = correct / len(y_test)

    # Under H0, correct predictions follow Binomial(n_test, 0.5)
    p_value = binomtest(
        correct,
        n=len(y_test),
        p=0.5,
        alternative="greater"
    ).pvalue

    return accuracy, p_value

#### Identical distributions

In [3]:
rng = np.random.default_rng(42)

n = 1000

P = rng.normal(loc=0, scale=1, size=n)
Q = rng.normal(loc=0, scale=1, size=n)

accuracy, p_value = c2st(P, Q)

print("Accuracy:", accuracy)
print("p-value:", p_value)

Accuracy: 0.52
p-value: 0.10872414660207075


They both are $P=N(0,1)$ , $Q=N(0,1)$

The null hypothesis is true: $H_0: P = Q$

The classifier got approximately $52 \% $ correct. That's close enough to random guessing, we don't have sufficient evidence that P and Q are different: we don't reject the null hypothesis. 


##### Different distributions

In [4]:
P = rng.normal(loc=0, scale=1, size=n)
Q = rng.normal(loc=3, scale=1, size=n)

accuracy, p_value = c2st(P, Q)

print("Accuracy:", accuracy)
print("p-value:", p_value)

Accuracy: 0.93
p-value: 7.103981575029298e-193


They are now different: $P=N(0,1)$ , $Q=N(3,1)$

C2ST accuracy was much greater than $0.50$, meaning the classifier successfully distinguished the two samples.
There is sufficient evidence against the null hypothesis, we reject it in favor of $H_1: P \neq Q$. 